# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates step-by-step how to explore and process the FAIR² dataset using the `mlcroissant` library. The dataset covers clinical, pathological, and molecular characteristics of second primary colorectal cancer in cancer survivors, provided in a FAIR Croissant schema.

### Dataset Source
- Croissant metadata URL: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant dataset URL 
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"Dataset: {getattr(dataset.metadata, 'name', '')}\n")
print(getattr(dataset.metadata, 'description', ''))

## 2. Data Overview

This section reviews record sets, their `@id`s, and available fields (and field `@id`s) present in the dataset. 

We'll use the Croissant metadata structure to inspect the dataset layout. All references to data entities (record sets, fields, etc.) will be made using their `@id` fields as required for reproducibility and clarity.

In [ ]:
# Inspect record sets by `@id` and their fields

record_set_objs = list(dataset.metadata.record_sets)
print(f"Number of record sets in dataset: {len(record_set_objs)}\n")
record_set_ids = []
for rs in record_set_objs:
    print(f"Record set: {rs.id}")
    record_set_ids.append(rs.id)
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.id} (name: {getattr(field, 'name', '')})")
    else:
        print("  No fields found.")
    print()
    # Optionally, look into columns within file objects, if useful
    if hasattr(rs, 'file_objects'):
        for fo in rs.file_objects:
            if hasattr(fo, 'columns'):
                print("  File object columns:")
                for c in fo.columns:
                    print(f"    - {getattr(c, 'id', '?')} (name: {getattr(c, 'name', '?')})")
        print()

if not record_set_ids:
    print("No record sets with fields found in the dataset metadata.")

## 3. Data Extraction

We'll load data from each available record set into pandas DataFrames for analysis.

All data is referenced by `@id`. Please identify the intended primary record set(s) for exploration from the overview above.

In [ ]:
# Collect all record set @id's
print("Record sets to extract:")
record_sets = record_set_ids  # From previous cell
print(record_sets)

dataframes = {}
# Download and load each record set to a DataFrame
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records for {record_set_id}")
    else:
        print(f"No records found for {record_set_id}\n")

# Preview columns and a sample from the first (primary) record set DataFrame
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nColumns in {first_rs}")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)

We'll perform simple data processing using the canonical primary record set. Typical processing steps include:
- Filtering out records based on a numeric field (e.g. age or interval variable),
- Normalizing numeric data,
- Grouping or summarizing data by a key attribute (e.g. sex or anatomical site),

Please modify the field `@id`s as needed for your analysis. All field/column references **must be made by their `@id`.**

In [ ]:
# Choose the main record set and a numeric field for analysis by `@id`

# Use the first loaded record set as example
record_set_id = list(dataframes.keys())[0] if dataframes else None

if record_set_id:
    df = dataframes[record_set_id]
    print(f"Working with record set {record_set_id} (rows: {len(df)})\n")
    # Show column (field) @id's and infer numeric fields
    print("Available fields (@id):")
    print(df.columns.tolist())

    # Identify numeric fields - here we try to infer them by pandas dtype
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field for analysis: {numeric_field_id}\n")

        threshold = df[numeric_field_id].mean()  # simple criterion: mean
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.1f} (total: {len(filtered_df)})")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' column:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Example grouping field - try to pick first string/categorical field aside from numeric_field_id
        group_field_candidates = [col for col in df.columns if col != numeric_field_id and df[col].dtype == 'object']
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            print(f"\nGrouping by field: {group_field_id}")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped.head())
        else:
            print("No suitable categorical grouping field found.")
    else:
        print("No numeric fields detected in record set.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize data distributions or relationships. Here, we'll generate a histogram of a numeric field, and, if possible, a bar plot grouped by a categorical field. All axis labels will identify the fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_fields:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Barplot by group (if available)
    if group_field_candidates:
        plt.figure(figsize=(8, 4))
        sns.barplot(data=df, x=group_field_id, y=numeric_field_id, ci=None)
        plt.title(f"Mean {numeric_field_id} per {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=40)
        plt.show()

## 6. Conclusion

- We explored and extracted the FAIR² dataset via its Croissant metadata package using `mlcroissant`, referencing all record sets and fields strictly by their `@id`.
- The notebook loaded the main record set, performed basic numeric filtering and normalization, and grouped the data by categorical attributes identified from the dataset schema.
- Simple visualizations were provided to understand field distributions and attribute relationships.

**Next steps:** You can extend this notebook for more sophisticated analysis (e.g., advanced clinical statistics, machine learning tasks, or cross-record set joins) by referencing all data elements via their Croissant `@id`s for traceable and reproducible workflows.